# FIUS-MoveSense: Data Exploration & Results
This notebook lets you explore the sensor data, features, and model results interactively.

In [ ]:
import sys
sys.path.insert(0, '..')
import config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import classification_report
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Processed Data

In [ ]:
import os
signals = np.load(os.path.join(config.PROCESSED_DIR, 'signals.npy'))
labels = np.load(os.path.join(config.PROCESSED_DIR, 'labels.npy'))
filtered = np.load(os.path.join(config.PROCESSED_DIR, 'filtered.npy'))
envelopes = np.load(os.path.join(config.PROCESSED_DIR, 'envelopes.npy'))
first_peaks = np.load(os.path.join(config.PROCESSED_DIR, 'first_peaks.npy'))

print(f'Signals: {signals.shape}')
print(f'Not Moving: {np.sum(labels==0)}, Moving: {np.sum(labels==1)}')

## 2. Signal Visualization

In [ ]:
# Plot one example from each class
t_ms = np.arange(signals.shape[1]) / config.SAMPLING_RATE * 1000

nm_idx = np.where(labels == 0)[0][0]
mv_idx = np.where(labels == 1)[0][0]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(t_ms, signals[nm_idx], linewidth=0.3, color='#2196F3')
axes[0].set_title('No Movement - Raw Signal', fontweight='bold')
axes[0].set_ylabel('ADC Value')
axes[0].grid(alpha=0.3)

axes[1].plot(t_ms, signals[mv_idx], linewidth=0.3, color='#E91E63')
axes[1].set_title('Movement - Raw Signal', fontweight='bold')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('ADC Value')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay filtered signal and envelope
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(t_ms, filtered[mv_idx], linewidth=0.3, color='#2196F3', label='Filtered')
ax.plot(t_ms, envelopes[mv_idx], linewidth=1, color='#E91E63', label='Envelope')
if first_peaks[mv_idx] >= 0:
    peak_t = first_peaks[mv_idx] / config.SAMPLING_RATE * 1000
    ax.axvline(peak_t, color='green', linestyle='--', label=f'First Peak ({peak_t:.2f} ms)')
ax.set_title('Filtered Signal with Envelope (Movement Example)', fontweight='bold')
ax.set_xlabel('Time (ms)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Feature Analysis

In [ ]:
df = pd.read_csv(os.path.join(config.FEATURES_DIR, 'features.csv'))
print(f'Feature matrix: {df.shape}')
print(f'\nClass distribution:\n{df["label"].value_counts()}')
df.head()

In [ ]:
# Feature distributions by class
top_features = ['percentile_25', 'zone_close_std', 'zone_background_std', 
                'zone_near_field_std', 'zero_crossing_rate', 'spectral_entropy']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, feat in zip(axes.flat, top_features):
    for label, color, name in [(0, '#2196F3', 'Not Moving'), (1, '#E91E63', 'Moving')]:
        ax.hist(df[df['label']==label][feat], bins=30, alpha=0.6, color=color, label=name)
    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)
plt.suptitle('Top Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (Random Forest)
importance = pd.read_csv(os.path.join(config.RESULTS_DIR, 'feature_importance.csv'))

fig, ax = plt.subplots(figsize=(10, 8))
top15 = importance.head(15)
ax.barh(range(len(top15)), top15['importance'].values, color='#2196F3')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Model Results

In [ ]:
results = pd.read_csv(os.path.join(config.RESULTS_DIR, 'evaluation_table.csv'))
print('Model Comparison:')
print(results.to_string(index=False))

In [ ]:
# Detailed classification report for best model
X_test = np.load(os.path.join(config.PROCESSED_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(config.PROCESSED_DIR, 'y_test.npy'))

best_model = joblib.load(os.path.join(config.MODELS_DIR, 'random_forest.joblib'))
y_pred = best_model.predict(X_test)

print('Classification Report (Random Forest):')
print(classification_report(y_test, y_pred, target_names=['Not Moving', 'Moving']))

In [ ]:
# Feature correlation heatmap
feature_cols = [c for c in df.columns if c != 'label']
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax, 
            xticklabels=True, yticklabels=True)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.xticks(fontsize=7, rotation=45, ha='right')
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()